# PoliMillionaire Game Tests

Run live games with selected local models, one after another.

No generated-answer API is used.


## Setup

Find the repo and local model cache.


In [65]:
import gc
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "polimillionaire" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the project folder.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
for path in [REPO_ROOT / "src", REPO_ROOT]:
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

os.environ.setdefault("HF_HOME", str(REPO_ROOT / "data" / "hf_home"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(REPO_ROOT / "data" / "hf_home" / "hub"))

print("Repo:", REPO_ROOT)
print("HF cache:", os.environ["HF_HOME"])


Repo: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire
HF cache: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/data/hf_home


## Settings

Pick models here. Each selected model gets its own game run.


In [66]:
RUN_API_CHECK = True
RUN_LIVE_GAME = True
PROMPT_FOR_CREDENTIALS = False

API_URL = "http://131.175.15.22:51111/"
COMPETITION_ID = 0

MODELS_TO_RUN = [
    {"label": "Gemma 4 E2B", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma 4 E4B", "model_id": "google/gemma-4-E4B-it", "run": False},
    {"label": "Qwen 2.5 3B", "model_id": "Qwen/Qwen2.5-3B-Instruct", "run": False},
    {"label": "Qwen 2.5 7B", "model_id": "Qwen/Qwen2.5-7B-Instruct", "run": True},
]

SAFE_DELAY_SECONDS = 2.0
ANSWER_TIMEOUT_SECONDS = 25.0

print("API check:", RUN_API_CHECK)
print("Live game:", RUN_LIVE_GAME)
print("Competition:", COMPETITION_ID)
print("Models:")
for item in MODELS_TO_RUN:
    print("-", item["label"], item["model_id"], "run=", item["run"])


API check: True
Live game: True
Competition: 0
Models:
- Gemma 4 E2B google/gemma-4-E2B-it run= False
- Gemma 4 E4B google/gemma-4-E4B-it run= False
- Qwen 2.5 3B Qwen/Qwen2.5-3B-Instruct run= False
- Qwen 2.5 7B Qwen/Qwen2.5-7B-Instruct run= True


## Login

Use environment variables, Colab secrets, or prompt mode.


In [67]:
import getpass
from dotenv import load_dotenv
from millionaire_client import AuthenticationError, MillionaireClient

# Load environment variables from .env file
load_dotenv()

USERNAME = os.environ.get("POLIMILLIONAIRE_USERNAME")
PASSWORD = os.environ.get("POLIMILLIONAIRE_PASSWORD")

try:
    from google.colab import userdata

    USERNAME = USERNAME or userdata.get("POLIMILLIONAIRE_USERNAME")
    PASSWORD = PASSWORD or userdata.get("POLIMILLIONAIRE_PASSWORD")
except Exception:
    pass

if PROMPT_FOR_CREDENTIALS and not USERNAME:
    USERNAME = input("PoliMillionaire username: ").strip()
if PROMPT_FOR_CREDENTIALS and not PASSWORD:
    PASSWORD = getpass.getpass("PoliMillionaire password: ")

print("Username configured:", bool(USERNAME))
print("Password configured:", bool(PASSWORD))


Username configured: True
Password configured: True


## Competitions

Turn on `RUN_API_CHECK` to list games.


In [68]:
client = None

if RUN_API_CHECK:
    if not USERNAME or not PASSWORD:
        raise RuntimeError("Set credentials first.")
    try:
        client = MillionaireClient(API_URL)
        user = client.login(USERNAME, PASSWORD)
        print("Logged in as", user.username)
        for competition in client.competitions.list_all():
            print(competition.id, competition.name, competition.max_levels)
    except AuthenticationError as exc:
        client = None
        print("Login failed:", exc)
else:
    print("API check skipped.")


Logged in as promptPotamo
0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15
4 Philosophy and Psychology 15
5 News 15


## Run Selected Models

Each model warms up, plays if enabled, then unloads.


In [69]:
# !pip install  -qU colab-xterm
# %load_ext colabxterm
# # It may ask you to restart, accept and run this cell again

### For Colab: Run the following commands on the terminal below:
To spawn the server deamon please run the cell with ```%xterm```. It will create a concurrent shell where we can paste the following commands but, at the same time, also run other cells!

```sudo apt-get install zstd```

```curl -fsSL https://ollama.com/install.sh | sh```

```ollama serve & ollama pull llama3```

In [70]:
# %xterm

In [ ]:
from dataclasses import dataclass, field
from typing import Any

@dataclass
class AnswerPrediction:
    option_id: int
    answer_text: str
    confidence: float | None = None
    reasoning: str | None = None
    metadata: dict[str, Any] = field(default_factory=dict)



In [ ]:
# INSTALL (run once if needed)

# !pip install -q \
#     langchain \
#     langchain-core \
#     langchain-community \
#     langchain-ollama \
#     wikipedia \
#     ddgs\
#     numexpr \
#     pydantic \
#     rank_bm25
#     trafilatura




# IMPORTS

import re
import logging
import math
from enum import Enum

import numexpr as ne
from ddgs import DDGS
from pydantic import BaseModel, Field

from dataclasses import dataclass, field
from typing import Any

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

# LOGGING (cleaner than prints)

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("trivia-agent")



# Small dataclasses shared by the assignment notebook helpers

@dataclass(frozen=True)
class AnswerOption:
    id: int
    text: str


@dataclass(frozen=True)
class Question:
    id: int
    text: str
    options: list[AnswerOption]
    level: int = 0
    metadata: dict[str, Any] = field(default_factory=dict)

    def valid_option_ids(self) -> set[int]:
        return {option.id for option in self.options}

    def first_option(self) -> AnswerOption:
        if not self.options:
            raise ValueError("Question has no answer options")
        return self.options[0]

    def get_option(self, option_id: int) -> AnswerOption | None:
        for option in self.options:
            if option.id == option_id:
                return option
        return None

    def require_option(self, option_id: int) -> AnswerOption:
        return self.get_option(option_id) or self.first_option()


# CONFIG

# MODEL_NAME = "qwen2.5:7b-instruct-q4_K_M"
MODEL_NAME = "gemma4:e4b"
TEMPERATURE = 0

MAX_DDG_RESULTS = 8
BM25_TOP_K = 4
TIMEOUT_DDG = 10
FETCH_TIMEOUT = 5


# OUTPUT SCHEMA (STRICT)

class TriviaAnswer(BaseModel):
    answer: int = Field(description="Option ID")
    confidence: float = Field(ge=0.00, le=1.00, description="Confidence score between 0.00 and 1.00, 2 decimals precision")
    evidence: str = Field(description="Short evidence-based justification")


class MemoryAttempt(BaseModel):
    answer: int = Field(description="Option ID")
    confidence: float = Field(ge=0.00, le=1.00, description="Confidence score between 0.00 and 1.00, 2 decimals precision")
    reasoning: str = Field(description="Short explanation of the reasoning behind the answer")


# ROUTING

class Route(str, Enum):
    CALCULATION = "calculation"
    RETRIEVAL = "retrieval"

class RouteDecision(BaseModel): 
    route: Route 


def route_question(question: str) -> Route:
    prompt = f""":
Classify the question into ONE route: 

1. CALCULATION 
Use ONLY if the question requires performing arithmetic, 
numerical computation, algebraic solving, or mathematical evaluation. 

2. RETRIEVAL 
Use for: 
- factual questions 
- definitions 
- conceptual explanations 
- historical questions 
- science questions 
- questions ABOUT formulas or theorems 
- explanatory questions

Question: 
{question}

Examples: 
Q: What is 25% of 320? 
A: CALCULATION 

Q: Solve 5x + 2 = 12 
A: CALCULATION 

Q: What is the formula for the volume of a cone?
A: RETRIEVAL 

Q: What is the Pythagorean theorem?
A: RETRIEVAL
"""
    result = router_llm.invoke(prompt)

    return result.route


# LLM

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
)

structured_llm = llm.with_structured_output(TriviaAnswer)

router_llm = llm.with_structured_output(RouteDecision)

memory_llm = llm.with_structured_output(MemoryAttempt)




MEMORY_PROMPT = """
You are a careful multiple-choice trivia solver.

Reasoning policy:
1. Answer the multiple-choice question using your internal knowledge only.
2. Compare ALL answer options carefully.
3. If uncertain, lower confidence to half.
4. If you dont know the answer, select the first option with the lowest confidence.
"""

REASONING_PROMPT_INIT = """QUESTION:
{question}

Return the most likely option.
Confidence score is a decimal number between 0.00 (lowest) and 1.00 (highest).

Answer:
"""


def reason_init(question: str) -> TriviaAnswer:
    prompt = ChatPromptTemplate.from_messages([
        ("system", MEMORY_PROMPT),
        ("human", REASONING_PROMPT_INIT)
    ])

    chain = prompt | memory_llm

    return chain.invoke({
        "question": question,
    })



# TOOLS

def fetch_readable_text(url: str, timeout: int = FETCH_TIMEOUT) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    resp = requests.get(url, headers=headers, timeout=timeout)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator=" ", strip=True)
    return text[:20000]  # cap size so one page doesn't dominate BM25

def ddgs_search_results(query: str, max_results: int = MAX_DDG_RESULTS) -> list[dict]:
    """Run DDGS search and return raw result dicts."""
    with DDGS(timeout=TIMEOUT_DDG) as ddgs:
        backend = "yandex,mojeek"
        results = list(ddgs.text(query, max_results=max_results, backend=backend))
    return results

def dedupe_results(results: list[dict]) -> list[dict]:
    """Deduplicate by normalized URL."""
    seen = set()
    deduped = []

    for r in results:
        url = (r.get("href") or "").strip()
        if not url:
            continue

        norm_url = url.lower().rstrip("/")
        if norm_url in seen:
            continue

        seen.add(norm_url)
        deduped.append(r)

    return deduped


def results_to_documents(results: list[dict]) -> list[Document]:
    """Convert DDGS results to LangChain Documents."""
    docs = []

    for r in results:
        title = r.get("title", "No title")
        snippet = r.get("body", "No snippet")
        url = r.get("href", "")

        if not url:
            continue

        try:
            fetched_text = fetch_readable_text(url)

            rank_text = "\n".join([
                title,
                snippet,
                fetched_text
            ]).strip()

            docs.append(
                Document(
                    page_content=rank_text,   # used only for BM25 scoring
                    metadata={
                        "title": title,
                        "url": url,
                        "snippet": snippet,   # keep original DDGS snippet
                    }
                )
            )
        except Exception:
            # fallback: still keep the result if fetching fails
            rank_text = "\n".join([title, snippet]).strip()

            docs.append(
                Document(
                    page_content=rank_text,
                    metadata={
                        "title": title,
                        "url": url,
                        "snippet": snippet,
                    }
                )
            )

    return docs



def rerank_with_bm25(query: str, docs: list[Document], top_k: int = BM25_TOP_K) -> list[Document]:
    """Rerank documents with BM25."""
    if not docs:
        return []

    retriever = BM25Retriever.from_documents(
        documents=docs,
        k=top_k,
    )

    return retriever.invoke(query)


def format_reranked_evidence(docs: list[Document]) -> str:
    """Format reranked docs for the reasoning prompt."""
    if not docs:
        return "No results found."

    blocks = []
    for i, doc in enumerate(docs, 1):
        blocks.append(
            f"TITLE: {doc.metadata.get('title', 'No title')}\n"
            f"URL: {doc.metadata.get('url', 'No URL')}\n"
            f"SNIPPET: {doc.metadata.get('snippet', 'No snippet')}"
        )

    return "\n\n".join(blocks)

def format_snippet_output(reranked_docs: list[Document]) -> str:
    if not reranked_docs:
        return "No results found."

    blocks = []
    for i, doc in enumerate(reranked_docs, 1):
        blocks.append(
            f"[{i}] TITLE: {doc.metadata.get('title', 'No title')}\n"
            f"URL: {doc.metadata.get('url', 'No URL')}\n"
            f"SNIPPET: {doc.metadata.get('snippet', 'No snippet')}"
        )

    return "\n\n".join(blocks)

@tool
def web_search_raw(query: str) -> list[dict]:
    """Web search for recent/time-sensitive facts."""
    try:
        results = ddgs_search_results(query, max_results=MAX_DDG_RESULTS)
        if not results:
            return "No results found."

        results = dedupe_results(results)
        return results
    except Exception as e:
        return [{"error": f"DDG search error: {e}"}]
    

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression.

    Supports:
    + - * / **
    sqrt()
    sin()
    cos()
    tan()

    IMPORTANT:
    Trigonometric functions use radians.
    """
    try:
        result = ne.evaluate(expression, local_dict={"pi": math.pi, "e": math.e})
        result = result.item()
        result = round(result, 4) if isinstance(result, float) else result
        return str(result)
    except Exception as e:
        return f"Calculator error: {e}"


TOOLS = [web_search_raw, calculator]

TOOL_MAP = {t.name: t for t in TOOLS}



# QUERY REWRITING 

def rewrite_query(question: str) -> str:
    prompt = f"""
Rewrite the following multiple-choice trivia question into a web search query.
- Preserve named entities, event names, titles, and dates.
- Focus mainly on the question stem, not the distractor options.
- Use one disambiguating keyword ONLY IF it helps retrieve evidence.
- Return ONLY the search query.

Question:
{question}
"""
    return llm.invoke(prompt).content.strip()



# EVIDENCE COMPRESSION

def compress_evidence(raw: str) -> str:
    if not raw:
        return ""

    prompt = f"""
Extract ONLY key factual points of this evidence:

{raw}

Return bullet points.
"""
    compressed = llm.invoke(prompt).content.strip()
    return compressed


# RETRIEVAL

def retrieve(query: str, compressed: bool = False) -> str:
    log.info(f"DDG query: {query}")

    raw_results = web_search_raw.invoke({"query": query})

    if isinstance(raw_results, list) and raw_results and "error" in raw_results[0]:
        return raw_results[0]["error"]

    docs = results_to_documents(raw_results)
    reranked_docs = rerank_with_bm25(query, docs, top_k=BM25_TOP_K)

    evidence = format_reranked_evidence(reranked_docs)

    evidence = evidence[:2500] # truncate to fit in prompt

    if compressed:
        evidence = compress_evidence(evidence)

    return evidence


# CALCULATION PIPELINE

def extract_expression(question: str) -> str:
    prompt = f"""
Extract ONLY the mathematical expression needed to solve the question.
Return ONLY a single-line expression compatible with Python's numexpr library. 
- Convert degrees to radians when necessary.
- Use 'pi' for π and 'e' for Euler's number.
- Allowed operators: +, -, *, /, **, %, &, |, ~

Examples:
123 * (456 / 789)
sqrt(144)
2 * (1 + 80 / 100)**10
tan(pi / 4) + e**2

Question:
{question}
"""
    expression = llm.invoke(prompt).content.strip()
    return expression


def run_math(question: str) -> str:
    expr = extract_expression(question)
    log.info(f"Math expression: {expr}")

    result = calculator.invoke({"expression": expr})
    evidence = f"Expression: {expr}\nResult: {result}"

    return evidence


# REASONING

def build_mcq(question: Question) -> str:
    options = "\n".join(f"{option.id}) {option.text}" for option in question.options)
    mcq_text = f"{question.text}\n{options}"
    return mcq_text


# SYSTEM_PROMPT = """
# You are a careful multiple-choice trivia solver.

# Rules:
# 1. Compare ALL answer options against the evidence.
# 2. Prioritize retrieved evidence over memorized knowledge.
# 3. Keep reasoning concise and factual.
# 4. If one option is only loosely related but another is explicitly stated, choose the explicit one.
# 5. If the evidence is ambiguous or conflicting, lower confidence.
# 6. The confidence score is a decimal number between 0.00 (lowest) and 1.00 (highest).
# 7. Base your answer on the question wording: answer what is being asked exactly.
# 8. Return exactly one option ID.
# """

SYSTEM_PROMPT = """
You are a careful multiple-choice trivia solver.

Reasoning policy:
1. Use retrieved evidence as the primary source when it is relevant and explicit.
2. Use your internal world knowledge to supplement incomplete evidence.
3. Compare ALL answer options carefully.
4. If retrieved evidence conflicts with internal knowledge:
   - prefer explicit retrieved evidence.
   - reduce confidence if uncertainty remains.
   - confidence must reflect certainty avoiding ambiguity.
5. If one option is only loosely related but another is explicitly stated, choose the explicit one.
6. Base your answer on the question wording: answer what is being asked exactly.
7. If you definitively don't know the answer, don't guess, select the first option with the lowest confidence.
8. Select EXACTLY ONE option ID.
"""


REASONING_PROMPT = """QUESTION:
{question}

EVIDENCE:
{evidence}


Return the most likely option.
Confidence score is a decimal number between 0.00 (lowest) and 1.00 (highest).

Answer:
"""

def reason(question: str, evidence: str) -> TriviaAnswer:
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", REASONING_PROMPT)
    ])

    chain = prompt | structured_llm

    return chain.invoke({
        "question": question,
        "evidence": evidence
    })


# MAIN PIPELINE


def ask_question(q: Question) -> TriviaAnswer:

    log.info("=== QUESTION ===")
    mcq_str = build_mcq(q)
    log.info(mcq_str)

    direct_result = reason_init(mcq_str)
    log.info("=== DIRECT ANSWER ATTEMPT ===")
    log.info(direct_result)

    if direct_result.confidence >= 0.95: 
        return direct_result

    route = route_question(mcq_str)
    log.info(f"ROUTE: {route}")

    # CALCULATION
    if route == Route.CALCULATION:
        evidence = run_math(q.text)

    # RETRIEVAL
    else:
        query = rewrite_query(mcq_str)
        evidence = retrieve(query)

    log.info("=== EVIDENCE ===")
    log.info(evidence[:1500])

    log.info("=== RESULT ===")
    result = reason(mcq_str, evidence)
    log.info(result)

    log.info("=== RESULT (STRUCTURED) ===")
    result_dict = result.model_dump_json(indent=2)
    log.info(result_dict)

    return result_dict


# EXAMPLES

# WARMUP QUESTION
warmup_question = Question(
    id=1,
    text="What is 2 + 2?",
    options=[
        AnswerOption(0, "3"),
        AnswerOption(1, "4"),
        AnswerOption(2, "5"),
        AnswerOption(3, "22"),
    ],
)


# RETRIEVAL QUESTION
factual_question = Question(
    id=2,
    text="Which country won the FIFA World Cup in 2018?",
    options=[
        AnswerOption(0, "Germany"),
        AnswerOption(1, "Argentina"),
        AnswerOption(2, "France"),
        AnswerOption(3, "Brazil"),
    ],
)


# RECENT QUESTION
recent_question = Question(
    id=3,
    text="Who won the UEFA Champions League in 2025?",
    options=[
        AnswerOption(0, "Real Madrid"),
        AnswerOption(1, "Manchester City"),
        AnswerOption(2, "PSG"),
        AnswerOption(3, "Bayern Munich"),
    ],
)


# CALCULATION QUESTION
calculation_question = Question(
    id=4,
    text="At an annual interest rate of 95%, what would 50 dollars be worth in 10 years?",
    options=[
        AnswerOption(0, "39748"),
        AnswerOption(1, "525"),
        AnswerOption(2, "3475"),
        AnswerOption(3, "500"),
    ],
)

math_question = Question(
    id=5,
    text="What is the result of cosine of 90 degrees ?",
    options=[
        AnswerOption(0, "45"),
        AnswerOption(1, "pi/2"),
        AnswerOption(2, "0"),
        AnswerOption(3, "1"),
    ],
)


question_1 = Question(
    id=6,
    text="What is the primary reason Mariah Carey was criticized for her performance on 'Dick Clark's New Year's Rockin' Eve' in 2017?",
    options=[
        AnswerOption(0, "She had technical difficulties with her in-ear monitors"),
        AnswerOption(1, "She lip-synced throughout the performance"),
        AnswerOption(2, "She performed an inappropriately inappropriate song"),
        AnswerOption(3, "She used auto-tune extensively"),
    ],
)

question_2 = Question(
    id=7,
    text="What was the name of the band Louis Armstrong played with that toured on the Mississippi River?",
    options=[
        AnswerOption(0, "The Dixie Jazz Band"),
        AnswerOption(1, "The Creole Jazz Band"),
        AnswerOption(2, "Fate Marable's Band"),
        AnswerOption(3, "Kid Ory's Band"),
    ],
)

question_3 = Question(
    id=8,
    text="How does the film The Babadook relate to Jennifer Kent's previous work?",
    options=[
        AnswerOption(0, "It is a short film she directed"),
        AnswerOption(1, "It is a remake of her previous film"),
        AnswerOption(2, "It is an extension of her short film Monster"),
        AnswerOption(3, "It is unrelated to her previous works"),
    ],
)

question_4 = Question(
    id=9,
    text="What term describes David Bowie's reinvention and visual presentation throughout his career?",
    options=[
        AnswerOption(0, "Plastic soul"),
        AnswerOption(1, "Glam rock"),
        AnswerOption(2, "Chameleon of rock"),
        AnswerOption(3, "Musical chameleon"),
    ],
)
question_5 = Question(
    id=10,
    text="The following best describes Clark Gable's role in World War II?",
    options=[
        AnswerOption(0, "He was a military general"),
        AnswerOption(1, "He remained in Hollywood and did not participate"),
        AnswerOption(2, "He was a spy for the United States"),
        AnswerOption(3, "He served in the United States Army Air Forces"),
    ],
)


result = ask_question(question_4)


INFO:trivia-agent:=== QUESTION ===
INFO:trivia-agent:What term describes David Bowie's reinvention and visual presentation throughout his career?
0) Plastic soul
1) Glam rock
2) Chameleon of rock
3) Musical chameleon
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:=== DIRECT ANSWER ATTEMPT ===
INFO:trivia-agent:answer=3 confidence=0.95 reasoning="The term 'musical chameleon' best describes David Bowie's reinvention and visual presentation throughout his career, as it accurately captures his ability to change styles and personas multiple times."


In [89]:
def bm25_preprocess(text: str) -> list[str]:
    """Simple tokenization for BM25."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()
MAX_DDG_RESULTS = 12
@tool
def web_search(query: str) -> str:
    """Web search for recent/time-sensitive facts."""
    try:
        with DDGS(timeout=TIMEOUT_DDG) as ddgs:
            results = ddgs.text(query, max_results=MAX_DDG_RESULTS, backend="wikipedia, brave, yandex, startpage, yahoo")

            if not results:
                return "No results found."
    
            docs = [
                Document(
                    page_content=f"TITLE: {r.get('title', 'No title')}\nSNIPPET: {r.get('body', 'No snippet')}",
                    metadata={
                        "title": r.get("title", ""),
                        "snippet": r.get("body", ""),
                        "url": r.get("href", ""),
                    }
                )
                for r in results
            ]
            
            print(f"Retrieved {len(docs)} documents from DDG for query: {query}")
            print(docs)

            retriever = BM25Retriever.from_documents(docs, k=3, preprocess_func=bm25_preprocess)

            reranked_docs = retriever.invoke(query)

            print(f"Reranked top {len(reranked_docs)} documents:")

            blocks = []
            for i, doc in enumerate(reranked_docs, 1):
                blocks.append(
                    f"[{i}] TITLE: {doc.metadata.get('title', 'No title')}\n"
                    f"URL: {doc.metadata.get('url', 'No URL')}\n"
                    f"SNIPPET: {doc.metadata.get('snippet', 'No snippet')}"
                )

            return "\n\n".join(blocks)
    except Exception as e:
        return f"DDG search error: {e}"
web_search.invoke({"query": "Mariah Carey criticism New Year's Rockin' Eve 2017 performance lip-syncing"})

INFO:primp:response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=Mariah%20Carey%20criticism%20New%20Year%27s%20Rockin%27%20Eve%202017%20performance%20lip-syncing 200
INFO:primp:response: https://search.brave.com/search?q=Mariah+Carey+criticism+New+Year%27s+Rockin%27+Eve+2017+performance+lip-syncing&source=web 200
INFO:primp:response: https://yandex.com/search/site/?text=Mariah+Carey+criticism+New+Year%27s+Rockin%27+Eve+2017+performance+lip-syncing&web=1&searchid=6430417 200


Retrieved 12 documents from DDG for query: Mariah Carey criticism New Year's Rockin' Eve 2017 performance lip-syncing
[Document(metadata={'title': 'Mariah Carey - Wikipedia', 'snippet': '"Mariah Carey\'s New Year\'s Eve Nightmare in Times Square".^ "Mariah Carey accused of lip-syncing during American Music Awards". Global News. October 11, 2018. Retrieved February 16, 2026.', 'url': 'https://en.wikipedia.org/wiki/Mariah_Carey'}, page_content='TITLE: Mariah Carey - Wikipedia\nSNIPPET: "Mariah Carey\'s New Year\'s Eve Nightmare in Times Square".^ "Mariah Carey accused of lip-syncing during American Music Awards". Global News. October 11, 2018. Retrieved February 16, 2026.'), Document(metadata={'title': 'The Worst New Year’s Eve Performance Ever Aired on Dick...', 'snippet': 'Mariah Carey\'s disastrous 2016 New Year\'s Eve performance left fans stunned. Technical difficulties derailed her set, but was sabotage involved?With no fix in sight, Carey half-heartedly lip-synced to "We Belong To

'[1] TITLE: Mariah Carey Suffers Epic Lip Sync Snafu on ‘New Year’s Rockin’ Eve’\nURL: https://www.hollywoodreporter.com/news/general-news/mariah-carey-suffers-epic-lip-sync-snafu-new-years-rockin-eve-960191/\nSNIPPET: January 1, 2017 - The singer was visibly frustrated when a vocal track malfunction set off her performance.\n\n[2] TITLE: Mariah Carey Times Square New Year\'s Eve performance: Singer\'s epic performance fail - CBS News\nURL: https://www.cbsnews.com/news/mariah-carey-times-square-new-years-eve-performance-fail/\nSNIPPET: January 1, 2017 - me watching mariah carey\'s performance pic.twitter.com/Z7a0FzXJDU— Magcon Follow Help(: (@CAMSPRlNCE) January 1, 2017 · This is not the first time Carey has been caught lip-syncing in public: In 2015, she was roundly mocked for lip-syncing during a performance at the Macy’s Thanksgiving Day parade.\n\n[3] TITLE: Mariah Carey Shakes off New Year\'s Eve Lip-Sync Meltdown: \'S*** Happens\'\nURL: https://www.nbcnews.com/news/us-news/mariah

In [90]:
import requests
import trafilatura

def fetch_markdown(url, timeout=15):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    resp = requests.get(url, headers=headers, timeout=timeout)
    resp.raise_for_status()

    return trafilatura.extract(
        resp.text,
        output_format="markdown",
        with_metadata=True
    )

# print(fetch_markdown("https://en.wikipedia.org/wiki/Pythagorean_theorem"))

In [98]:
from abc import ABC, abstractmethod
from typing import Any, Protocol
from src.polimillionaire.strategies import parse_answer_prediction

class BaseStrategy(ABC):
    name = "base"

    @abstractmethod
    def answer(self, question: Question) -> AnswerPrediction:
        raise NotImplementedError
    
class LocalLLM(Protocol):
    model_name: str

    def generate(self, prompt: str, **kwargs: object) -> str:
        ...

class OllamaStrategy(BaseStrategy):
    name = "Ollama-Gwen2.5-7b"

    def __init__(self):
        self.llm = ChatOllama(model="gwen2.5:7b", temperature=0)

    def answer(self, question: Question) -> AnswerPrediction:
        prediction = ask_question(question)

        # raw_text = self.llm.generate(build_prompt(question))
        prediction = parse_answer_prediction(str(prediction), question, strategy_name=self.name)
        prediction.metadata["strategy"] = self.name
        prediction.metadata["model_name"] = getattr(self.llm, "model_name", "unknown")
        prediction.metadata["device"] = getattr(self.llm, "device_summary", "unknown")
        return prediction

In [ ]:
from datetime import datetime, timezone

from polimillionaire.runner import GameRunner, RunLogger
from polimillionaire.strategies import GemmaLLMConfig, GemmaStrategy
from polimillionaire.types import AnswerOption, Question

warmup_question = Question(
    id=1,
    text="What is 2 + 2?",
    options=[
        AnswerOption(0, "3"),
        AnswerOption(1, "4"),
        AnswerOption(2, "5"),
        AnswerOption(3, "22"),
    ],
)


def clear_model_memory():
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        if hasattr(torch, "mps") and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()
    except Exception as exc:
        print("Cleanup warning:", exc)


def make_strategy(model_id: str) -> GemmaStrategy:
    config = GemmaLLMConfig(
        model_id=model_id,
        inference_backend="auto_model",
        device_map="auto",
        dtype="auto",
        max_new_tokens=8,
        temperature=0.0,
        do_sample=False,
        num_beams=1,
        seed=42,
        generation_max_time_seconds=18.0,
        timeout_seconds=120.0,
    )
    return GemmaStrategy(model_config=config)


def clean_name(label: str) -> str:
    return "_".join(label.lower().split())


if RUN_LIVE_GAME and (not USERNAME or not PASSWORD):
    raise RuntimeError("Set credentials first.")
if RUN_LIVE_GAME and client is None:
    client = MillionaireClient(API_URL)
    client.login(USERNAME, PASSWORD)

run_results = []
for item in MODELS_TO_RUN:
    if not item.get("run", False):
        print("Skipped", item["label"])
        continue

    strategy = None
    try:
        print()
        print("Model:", item["label"])
        # strategy = make_strategy(item["model_id"])
        strategy = OllamaStrategy()

        warmup = strategy.answer(warmup_question)
        
        print("warmup option:", warmup.option_id, warmup.answer_text)
        print("fallback:", warmup.metadata.get("fallback"))
        print("device:", warmup.metadata.get("device"))
        if warmup.metadata.get("fallback"):
            raise RuntimeError(f"Warm-up failed for {item['label']}.")

        result = {"label": item["label"], "model_id": item["model_id"], "warmup_option": warmup.option_id}

        if RUN_LIVE_GAME:
            run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
            log_path = REPO_ROOT / "results" / "runs" / f"{clean_name(item['label'])}_{run_id}.jsonl"
            runner = GameRunner(
                client=client,
                safe_delay_seconds=SAFE_DELAY_SECONDS,
                answer_timeout_seconds=ANSWER_TIMEOUT_SECONDS,
                logger=RunLogger(log_path),
            )
            game = runner.play(COMPETITION_ID, strategy)
            result.update(
                {
                    "current_level": game.current_level,
                    "earned_amount": game.earned_amount,
                    "log_path": str(log_path),
                }
            )
            print("Reached level:", game.current_level)
            print("Earned amount:", game.earned_amount)
            print("Log path:", log_path)
        else:
            print("Live game skipped for", item["label"])

        run_results.append(result)
    finally:
        del strategy
        clear_model_memory()
        print("Cleared model memory after", item["label"])

run_results


INFO:trivia-agent:=== QUESTION ===
INFO:trivia-agent:What is 2 + 2?
0. 3
1. 4
2. 5
3. 22


Skipped Gemma 4 E2B
Skipped Gemma 4 E4B
Skipped Qwen 2.5 3B

Model: Qwen 2.5 7B


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:=== DIRECT ANSWER ATTEMPT ===
INFO:trivia-agent:answer=1 confidence=0.95 reasoning='The question asks for the sum of 2 + 2. The correct answer is 4. Option 1 (4) is clearly the right choice among the given options.'
INFO:trivia-agent:=== QUESTION ===
INFO:trivia-agent:How does the film 'The Shawshank Redemption' primarily convey its message of hope?
0. By Andy's escape from the prison
1. By Red's final speech
2. Through the prison's harsh environment
3. Through explicit dialogue


warmup option: 1 4
fallback: False
device: unknown


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:=== DIRECT ANSWER ATTEMPT ===
INFO:trivia-agent:answer=1 confidence=0.65 reasoning="Red's final speech conveys the message of hope through his reflections and the release letter Andy mailed to him, symbolizing freedom and hope even in the most oppressive environments. While Andy's escape is a pivotal moment, it might be seen as more of an action than a primary method for conveying the film's overarching theme of hope. The prison environment and explicit dialogue are also elements but not as explicitly focused on hope as Red’s speech."
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:ROUTE: Route.RETRIEVAL
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:trivia-agent:DDG query: how does the film The Shawshank Redemption primarily convey its message of hope through scenes or themes
INFO:primp:response: https

Reached level: 3
Earned amount: 200
Log path: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/results/runs/qwen_2.5_7b_20260526_202542.jsonl
Cleared model memory after Qwen 2.5 7B


[{'label': 'Qwen 2.5 7B',
  'model_id': 'Qwen/Qwen2.5-7B-Instruct',
  'warmup_option': 1,
  'current_level': 3,
  'earned_amount': 200,
  'log_path': '/Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/results/runs/qwen_2.5_7b_20260526_202542.jsonl'}]

INFO:trivia-agent:answer=3 confidence=0.9 evidence="The evidence from multiple sources, including Wikipedia and Hertel Walls, clearly states that after Humphrey Bogart's death in 1957, Frank Sinatra became the leader of the Rat Pack. This information is explicitly supported by the snippets provided."
INFO:trivia-agent:=== RESULT (STRUCTURED) ===
INFO:trivia-agent:{
  "answer": 3,
  "confidence": 0.9,
  "evidence": "The evidence from multiple sources, including Wikipedia and Hertel Walls, clearly states that after Humphrey Bogart's death in 1957, Frank Sinatra became the leader of the Rat Pack. This information is explicitly supported by the snippets provided."
}


## Results

Read recent game logs.


In [ ]:
from polimillionaire.runner import load_jsonl, summarize_attempts

run_logs = sorted((REPO_ROOT / "results" / "runs").glob("*.jsonl"), key=lambda path: path.stat().st_mtime)

for path in run_logs[-5:]:
    print(path.name)

if run_logs:
    rows = load_jsonl(run_logs[-1])
    print("latest:", run_logs[-1].name)
    print(summarize_attempts(rows))
else:
    print("No logs found.")


## Done

Use `run_results` and the JSONL logs for comparison.
